In [ ]:
!pip uninstall -y trl sentence-transformers
!pip install -q transformers peft bitsandbytes accelerate datasets


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import transformers
import peft

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


Transformers: 4.57.6
PEFT: 0.18.1
Torch: 2.9.0+cu126
CUDA: True


In [ ]:
raw_dataset = load_dataset("zou-lab/MedCaseReasoning")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/897 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/99.3M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/7.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13092 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/897 [00:00<?, ? examples/s]

In [ ]:
def build_text(example):
    return {
        "text": (
            "You are a medical expert.\n\n"
            "Task:\n"
            "1. Identify the most likely diagnosis.\n"
            "2. List ALL clinical reasons used to reach the diagnosis.\n"
            "3. Be exhaustive. Do not summarize.\n\n"
            "Case:\n"
            f"{example['case_prompt']}\n\n"
            "Final Diagnosis:\n"
            f"{example['final_diagnosis']}\n\n"
            "Clinical Reasons (exhaustive list):\n"
            f"{example['diagnostic_reasoning']}\n\n"
            "Differential Diagnoses:\n"
        )
    }

train_text = raw_dataset["train"].map(
    build_text,
    remove_columns=raw_dataset["train"].column_names,
)

Map:   0%|          | 0/13092 [00:00<?, ? examples/s]

In [ ]:
MODEL_ID = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


tokenizer_config.json:   0%|          | 0.00/267 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
MAX_LENGTH = 1024

def tokenize(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_train = train_text.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)


Map:   0%|          | 0/13092 [00:00<?, ? examples/s]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


config.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()


trainable params: 5,242,880 || all params: 2,599,490,560 || trainable%: 0.2017


In [ ]:
from transformers import TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
import os

# Output directory on Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/biomedlm-qlora-rr"

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    num_train_epochs=1,

    fp16=True,

    logging_steps=50,

    save_steps=500,
    save_strategy="steps",
    save_total_limit=2,

    report_to="none",

    gradient_checkpointing=False,  # MUST be False for 4-bit QLoRA
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
)

# 🔑 Auto-resume logic (SAFE)
last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found. Starting training from scratch.")
    trainer.train()


Resuming training from checkpoint: /content/drive/MyDrive/biomedlm-qlora-rr/checkpoint-2500


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
2550,2.379800
2600,2.197400
2650,2.170000
2700,2.165500
2750,2.204300
2800,2.185200
2850,2.171400
2900,2.186700
2950,2.153300
3000,2.145600


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
